# 04 · Geographical data visualization

*Notebooks: [01 Overview](01-overview.ipynb) · [02 Formats](02-formats.ipynb) · [03 Data](03-data.ipynb) · [04 Maps](04-maps.ipynb) · [05 Plots](05-plots.ipynb) · [06 Compute](06-compute.ipynb)*

`autk-map` draws 2D and 3D layers with WebGPU straight from GeoJSON, with no tile server and no backend. In the
JavaScript tutorial you manage a map's lifecycle (`new AutkMap`, `init`, `loadCollection`, `draw`). In pyautark a `Map`
is a view of the spec, and the engine does the lifecycle. What stays the same is the model: a map has **layers**, each
a table plus how to draw it.

In [ ]:
import autark as ak
import tutorial as t

In [ ]:
neighborhoods = ak.GeoJSON("neighborhoods", url=t.url("data/map/mnt_neighs.geojson"))
noise = ak.CSV("noise", url=t.url("data/map/mnt_noise.csv"), geometry=ak.latlng("latitude", "longitude"))

## 1 The map

`ak.Map(layers=[...])` is one map. `name` identifies it (links use it), `camera=ak.Camera(zoom, pitch, bearing)` frames
it, and `style` picks a base style. A **layer** names its table and gets a title (`id`) that the legend, links and the
layer menu use.

In [ ]:
ak.Spec(
    data=[neighborhoods],
    views=[ak.Map(
        name="manhattan",
        camera=ak.Camera(pitch=40, bearing=20),
        layers=[ak.Layer(neighborhoods, id="neighborhoods").style(color="#9ecae9", opacity=0.9)],
    )],
).widget()

## 2 Several layers

Layers draw in order, the first at the bottom. Style a layer with `.style(color=..., opacity=..., strokeColor=...,
width=..., size=...)`: `width` is the width of lines, `size` the radius of points, both in meters, and `strokeColor`
outlines polygons. Colors are `#rrggbb`.

In [ ]:
ak.Spec(
    data=[neighborhoods, noise],
    views=[ak.Map(layers=[
        ak.Layer(neighborhoods, id="neighborhoods").style(color="#dddddd", strokeColor="#999999"),
        ak.Layer(noise, id="noise").style(color="#d62728", opacity=0.5, size=40),
    ])],
).widget()

## 3 Connecting OSM layers

An OSM source gives roads, buildings and more as separate layers, drawn with Autark's physical styles: roads by class,
buildings in 3D, water and parks in their colors. Give each layer a title so the layer menu names it.

In [ ]:
osm = ak.OSM(
    "mnt", bbox=(-74.02, 40.700, -73.995, 40.72),
    layers=["surface", "water", "parks", "roads", "buildings"],
    source="pbf", pbf_file_url=t.url("data/osm/lower_mnt.osm.pbf"),
)

ak.Spec(
    data=[osm],
    views=[ak.Map(
        camera=ak.Camera(pitch=50),
        layers=[ak.Layer(osm.layer_source(layer), id=layer) for layer in osm.layers],
    )],
).widget()

## 4 Raster layers

Rasters are `GeoTIFF` sources or `Heatmap` tables. A raster layer is colored by its value, and the legend shows the
range. The map is not framed by a raster, so put a vector layer under it (below, the neighborhoods).

In [ ]:
heat = ak.Heatmap(noise, output="heat", near=ak.Near(800), grid=ak.HeatmapGrid(rows=40, columns=40), group_by=[ak.count()])

ak.Spec(
    data=[neighborhoods, noise],
    transforms=[heat],
    views=[ak.Map(layers=[
        ak.Layer(neighborhoods, id="neighborhoods").style(color="#f0f0f0"),
        ak.Layer("heat", id="noise density").encode(color=ak.field("count.noise", ak.Scale(scheme="magma"))).style(opacity=0.75),
    ])],
).widget()

## 5 Thematic data

A layer is colored by a property with `.encode(color=ak.field(path, scale))`. The `Scale` says how:

- `type="quantile"` (equal counts per color) or the default linear scale; `domain_strategy="percentile"` clips outliers;
- `scheme`: a color ramp by name (`"blues"`, `"viridis"`, `"spectral"`, `"redblue"`...) or a categorical scheme
  (`"tableau10"`, `"set1"`, `"observable10"`...);
- `domain=[low, high]` with `domain_strategy="user"` fixes the range, and for `type="categorical"` a list of values.

### 5.1 Categorical attributes

Road classes are text: color them with a categorical scheme. The property is `highway`.

In [ ]:
roads = ak.OSM("mnt", bbox=(-74.02, 40.700, -73.995, 40.72), layers=["roads"],
               source="pbf", pbf_file_url=t.url("data/osm/lower_mnt.osm.pbf"))

ak.Spec(
    data=[roads],
    views=[ak.Map(layers=[
        ak.Layer("mnt_roads", id="road classes").encode(color=ak.field("highway", ak.Scale(type="categorical", scheme="tableau10"))),
    ])],
).widget()

### 5.2 Diverging attributes

A diverging scheme has a neutral middle. Color the neighborhoods by area with `"spectral"`, quantile-binned:

In [ ]:
ak.Spec(
    data=[neighborhoods],
    views=[ak.Map(layers=[
        ak.Layer(neighborhoods, id="area").encode(
            color=ak.field("shape_area", ak.Scale(type="quantile", scheme="spectral"))
        ),
    ])],
).widget()

### 5.3 Spatial joins

Join counts first (notebook 03), then color by the new property. `sjoin.count.noise` is the number of noise complaints
in each neighborhood.

In [ ]:
ak.Spec(
    data=[neighborhoods, noise],
    transforms=[ak.SpatialJoin(root=neighborhoods, join=noise, group_by=[ak.count()])],
    views=[ak.Map(layers=[
        ak.Layer(neighborhoods, id="complaints").encode(color=ak.field("sjoin.count.noise", ak.Scale(type="quantile", scheme="reds"))),
    ])],
).widget()

## 6 Interactions

A map is interactive. Double-click a feature to pick it once picking is on for its layer (the layer menu has a toggle).
To connect a map to a plot, give the plot a **selection** and add a **link**; that is the subject of the next notebook.
Selecting in the plot highlights the same features on the map, and picking on the map selects them in the plot.

```python
brush = ak.interval("brush")
spec = ak.Spec(
    views=[ak.Map(...layers with id="neighborhoods"...), ak.Scatterplot(..., selection=brush)],
    links=[ak.Link(brush, target="neighborhoods")],   # highlight on the layer titled "neighborhoods"
)
```